# C.3 lr_pull / n_consolidation_events sweep — parallel CUDA (Colab)

**Purpose.** Drill-down on the 2026-05-26 Path α null at the Phase 2 synthetic operating point. The precommit running log names two candidate explanations for the null: (1) consolidation is too weak to express corpus-specific learning at this scale, (2) synthetic vocab=200 is too thin. This notebook tests (1) by sweeping `lr_pull` and `n_consolidation_events`. If any cell breaks the null, escalate to n=10. If all stay null, this strengthens the case for pivoting to Path β (WikiText-2).

**This is a DRILL-DOWN, not a graduation experiment.** n_seeds=3; C.3 graduation gate requires n≥10.

**Headline metric (per [phase-3-deep-dive.md:180-189](https://github.com/Dypatterson/Neuro-AI/blob/main/notes/emergent-codebook/phase-3-deep-dive.md)):** Regime-stratified Recall@K vs. genuine shuffled-token control.

**Anchor:** `reports/c3_smoke_alpha_anti_2026-05-26/` (CPU). This notebook re-runs baseline + 4 sweeps on CUDA so all 5 are device-matched.

| Cell | lr_pull | n_events |
|---|---:|---:|
| baseline | 0.1 | 1000 |
| A | 0.5 | 1000 |
| B | 1.0 | 1000 |
| C | 0.1 | 3000 |
| D | 1.0 | 3000 |

**Reliability notes for this revision:**
- Cell 1 patch verification includes pre- and post-line-count checks for both files.
- Cell 1 applies **two** patches: (a) `--lr-pull`/`--lr-push` CLI flags in the C.3 driver, (b) CUDA-eigvalsh degeneracy fix in `src/energy_memory/phase4/consolidation.py` — cuSOLVER raises `LinAlgError 4095` on the low-rank high-D σ in `_spatial_bimodality_signal`, so the fix falls back to CPU LAPACK only when CUDA fails.
- Cell 6 runs **one process synchronously first as a smoke test** and dumps its log if it fails — surfaces startup errors before the parallel launch.
- Cell 7 uses `sys.executable` (not `python`) to guarantee interpreter match.
- Cell 7 staggers subprocess launches by 1.5 s to avoid simultaneous CUDA-init races.
- Cell 7 auto-prints the last 50 lines of each failed subprocess's log.

In [ ]:
# 1. Clone the repo and apply both patches (CLI flags + kernel-trick eigvalsh fix).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout codex/phase5-prime-bundle-first-scene-memory
!git log --oneline -3

# File-state baseline (line counts + tracer-string presence) BEFORE patching.
import subprocess
pre_lines_driver = int(subprocess.check_output(['wc', '-l', 'experiments/c3_phase3_exit_criterion.py']).decode().split()[0])
pre_lines_cons   = int(subprocess.check_output(['wc', '-l', 'src/energy_memory/phase4/consolidation.py']).decode().split()[0])
pre_pull = subprocess.run(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py'], capture_output=True, text=True).stdout.strip()
pre_gram = subprocess.run(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py'], capture_output=True, text=True).stdout.strip()
print(f'pre-patch: driver {pre_lines_driver} lines, consolidation {pre_lines_cons} lines')
print(f'pre-patch tracers: --lr-pull hits = {pre_pull}; kernel-trick gram fix hits = {pre_gram}')

patch = r'''diff --git a/experiments/c3_phase3_exit_criterion.py b/experiments/c3_phase3_exit_criterion.py
--- a/experiments/c3_phase3_exit_criterion.py
+++ b/experiments/c3_phase3_exit_criterion.py
@@ -576,6 +576,8 @@ def _run_single_seed_condition(
     k: int,
     alpha_anti: float,
     repulsion_step_size: float,
+    lr_pull: float,
+    lr_push: float,
     device: str,
     repo_root: Path,
     wikitext_corpus: Optional[_WikiTextCorpus] = None,
@@ -756,6 +758,8 @@ def _run_single_seed_condition(
             vocab_size=vocab_size,
             n_events=n_consolidation_events,
             device=device,
+            lr_pull=lr_pull,
+            lr_push=lr_push,
             repulsion_step_size=repulsion_step_size,
         )
 
@@ -838,6 +842,8 @@ def run(
     n_consolidation_events: int = 1000,
     alpha_anti: float = 0.0,
     repulsion_step_size: float = 0.0,
+    lr_pull: float = 0.1,
+    lr_push: float = 0.05,
     device: str,
     output_dir: Path,
     repo_root: Path,
@@ -919,6 +925,8 @@ def run(
                     k=k,
                     alpha_anti=alpha_anti,
                     repulsion_step_size=repulsion_step_size,
+                    lr_pull=lr_pull,
+                    lr_push=lr_push,
                     device=device,
                     repo_root=repo_root,
                     wikitext_corpus=wikitext_corpus,
@@ -1010,6 +1018,8 @@ def run(
             "substrate_repulsion_active": bool(
                 alpha_anti > 0.0 and repulsion_step_size > 0.0
             ),
+            "lr_pull": float(lr_pull),
+            "lr_push": float(lr_push),
             "operating_point": {
                 "D": D,
                 "landscape_size": landscape_size,
@@ -1370,6 +1380,27 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
             "smoke (no inter-atom-separability force)."
         ),
     )
+    parser.add_argument(
+        "--lr-pull",
+        type=float,
+        default=0.1,
+        help=(
+            "Per-event consolidation pull learning rate (OnlineCodebookUpdater "
+            "lr_pull). Default 0.1 matches the existing Path α smoke. Sweep "
+            "above this to test whether consolidation strength is too weak "
+            "to express corpus-specific learning at the synthetic operating "
+            "point."
+        ),
+    )
+    parser.add_argument(
+        "--lr-push",
+        type=float,
+        default=0.05,
+        help=(
+            "Per-event consolidation push learning rate (OnlineCodebookUpdater "
+            "lr_push). Default 0.05 matches the existing Path α smoke."
+        ),
+    )
     parser.add_argument(
         "--repulsion-step-size",
         type=float,
@@ -1468,6 +1499,8 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
         n_consolidation_events=args.n_consolidation_events,
         alpha_anti=args.alpha_anti,
         repulsion_step_size=args.repulsion_step_size,
+        lr_pull=args.lr_pull,
+        lr_push=args.lr_push,
         device=args.device,
         output_dir=output_dir,
         repo_root=repo_root,
diff --git a/src/energy_memory/phase4/consolidation.py b/src/energy_memory/phase4/consolidation.py
--- a/src/energy_memory/phase4/consolidation.py
+++ b/src/energy_memory/phase4/consolidation.py
@@ -639,10 +639,29 @@ class ConsolidationState:
         # Hermitian Gram of centered basin members. For complex (FHRR)
         # tensors, diffs.conj().T @ diffs is Hermitian → real eigenvalues
         # via torch.linalg.eigh.
-        sigma = (diffs.conj().transpose(-1, -2) @ diffs) / float(n)
+        # Compute the eigenvalues of σ = diffs.conj().T @ diffs / n via the
+        # n×n Gram matrix gram = diffs @ diffs.conj().T / n instead of the
+        # D×D scatter matrix. The two matrices share exactly the same set
+        # of non-zero eigenvalues (standard "kernel trick" identity); the
+        # D×D form additionally carries (D - n) trivial zero eigenvalues
+        # because rank(σ) ≤ n_members ≤ basin_trace_buffer_size (64) ≪ D
+        # (4096 by default in this project). That (D - n) zero subspace
+        # makes σ numerically ill-conditioned at the precision available
+        # to torch.linalg.eigvalsh — observed on Colab CUDA at 2026-05-27
+        # as LinAlgError 4095 and even on CPU LAPACK as LinAlgError 5/12.
+        # The n×n Gram path is full-rank for non-degenerate samples and
+        # an order of magnitude smaller (4 KB vs 16 MB at D=4096, n=8).
+        # Mathematically byte-identical at the λ_1 / λ_2 layer used below;
+        # the C.2.2 dynamic's behavior is unchanged.
+        gram = (diffs @ diffs.conj().transpose(-1, -2)) / float(n)
         # Eigh returns ascending eigenvalues. Take top two: λ_1 (last),
         # λ_2 (second-to-last). All ops stay on-device.
-        eigvals = torch.linalg.eigvalsh(sigma)
+        try:
+            eigvals = torch.linalg.eigvalsh(gram)
+        except torch._C._LinAlgError:
+            # Defensive: keep the CPU fallback in case some pathological
+            # input still trips cuSOLVER (e.g. identical basin members).
+            eigvals = torch.linalg.eigvalsh(gram.cpu()).to(gram.device)
         lam_1 = eigvals[-1]
         lam_2 = eigvals[-2] if eigvals.shape[0] >= 2 else torch.zeros_like(lam_1)
         # Clamp at 0 — eigh may return tiny negatives for near-singular Σ.
'''

with open('/tmp/c3_combined.patch', 'w') as f:
    f.write(patch)
check = subprocess.run(['git', 'apply', '--check', '/tmp/c3_combined.patch'], capture_output=True, text=True)
if check.returncode == 0:
    subprocess.check_call(['git', 'apply', '/tmp/c3_combined.patch'])
    print('combined patch applied via git apply.')
else:
    if int(pre_pull or '0') >= 1 and int(pre_gram or '0') >= 1:
        print('both patches already in branch — skipping apply.')
    else:
        print('PATCH APPLY FAILED — stderr below:')
        print(check.stderr)
        # Try the patches one-at-a-time so partial state is debuggable.
        print('\n--- attempting per-hunk debug check ---')
        check2 = subprocess.run(['git', 'apply', '--check', '--verbose', '/tmp/c3_combined.patch'], capture_output=True, text=True)
        print(check2.stderr)
        raise SystemExit('Cannot continue without both patches.')

post_pull = subprocess.check_output(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_push = subprocess.check_output(['grep', '-c', '--', '--lr-push', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_gram = subprocess.check_output(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py']).decode().strip()
print(f'post-patch tracers: --lr-pull = {post_pull}; --lr-push = {post_push}; kernel-trick gram fix = {post_gram}')
assert int(post_pull) >= 1 and int(post_push) >= 1, 'CLI flags missing after patch'
assert int(post_gram) >= 1, 'kernel-trick gram fix missing after patch'

In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted; results target = /content/drive/MyDrive/neuro-ai/results/c3_sweep_2026-05-27/')

In [ ]:
# 3. Verify Python interpreter + deps + CUDA availability (parent doesn't init CUDA here).
import sys, subprocess
print('sys.executable:', sys.executable)
print('python version:', sys.version.split()[0])
print('which python:', subprocess.run(['which', 'python'], capture_output=True, text=True).stdout.strip())
print('which python3:', subprocess.run(['which', 'python3'], capture_output=True, text=True).stdout.strip())

# torch is preinstalled on Colab; verify version compatibility.
import torch, numpy as np
print(f'torch: {torch.__version__} | numpy: {np.__version__} | cuda available: {torch.cuda.is_available()}')
# Quick CUDA test WITHOUT init in the parent — query property, don't allocate.
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'device 0 name: {torch.cuda.get_device_name(0)}')

In [ ]:
# 4. CPU-ONLY sanity check on the substrate. Parent must NOT touch CUDA before launching workers.
import sys, gc
sys.path.insert(0, '/content/Neuro-AI/src')
from energy_memory.substrate.torch_fhrr import TorchFHRR
_sub = TorchFHRR(dim=64, seed=0, device='cpu')
_cb = _sub.random_vectors(5)
print('substrate ok:', _cb.shape, _cb.dtype, _cb.device)
del _sub, _cb
gc.collect()

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE TEST — one subprocess synchronously with the smallest config.
#    If this fails we get the error in-cell instead of debugging 5 silent failures.
import subprocess, sys, os
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'

smoke_out = Path('reports/c3_smoke_test_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/c3_smoke_test.log')

cmd = [
    sys.executable, 'experiments/c3_phase3_exit_criterion.py',
    '--seeds', '0',
    '--device', 'cuda',
    '--lr-pull', '0.1',
    '--lr-push', '0.05',
    '--n-consolidation-events', '50',  # tiny — runs in <2 min
    '--alpha-anti', '0.01',
    '--repulsion-step-size', '0.05',
    '--output-dir', str(smoke_out),
]
print('cmd:', ' '.join(cmd))
print()
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)

print(f'smoke exit code: {rc}')
print(f'json written: {(smoke_out / "c3_summary.json").exists()}')
print()
print('=== smoke log (last 80 lines) ===')
!tail -80 {smoke_log}

if rc != 0:
    raise SystemExit('Smoke test failed — fix before launching parallel sweep.')
print()
print('Smoke OK. Proceed to parallel launch.')

In [ ]:
# 7. PARALLEL launch of 5 cells. Staggered 1.5s; auto-prints log tail on failure.
import subprocess, os, time, signal, sys
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

CELLS = [
    ('baseline_lrpull01_n1000', 0.1, 1000),
    ('lrpull05_n1000',          0.5, 1000),
    ('lrpull10_n1000',          1.0, 1000),
    ('lrpull01_n3000',          0.1, 3000),
    ('lrpull10_n3000',          1.0, 3000),
]

log_root = Path('reports/c3_sweep_colab_logs')
log_root.mkdir(parents=True, exist_ok=True)

def launch(tag, lr_pull, n_events):
    out_dir = f'reports/c3_sweep_{tag}_2026-05-27'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}.log'
    logf = open(log_path, 'w')
    proc = subprocess.Popen(
        [PY, 'experiments/c3_phase3_exit_criterion.py',
         '--seeds', '0,1,2',
         '--device', 'cuda',
         '--lr-pull', str(lr_pull),
         '--lr-push', '0.05',
         '--n-consolidation-events', str(n_events),
         '--alpha-anti', '0.01',
         '--repulsion-step-size', '0.05',
         '--output-dir', out_dir],
        stdout=logf, stderr=subprocess.STDOUT,
    )
    return proc, logf, out_dir, log_path

def snapshot(remaining, t0):
    elapsed = (time.time() - t0) / 60
    print(f'  --- snapshot at {elapsed:.1f} min ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    for tag, (proc, logf, out_dir, log_path) in remaining.items():
        size = log_path.stat().st_size if log_path.exists() else 0
        try:
            line = subprocess.check_output(['tail', '-1', str(log_path)],
                stderr=subprocess.DEVNULL).decode().strip()[:90]
        except Exception:
            line = ''
        print(f'  {tag:>28}: {size:>7}b  {line}')

def kill_all(remaining):
    for tag, (proc, logf, out_dir, log_path) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL)
            logf.close()
        except Exception:
            pass

# Staggered launch.
procs = {}
for tag, lp, ne in CELLS:
    procs[tag] = launch(tag, lp, ne)
    time.sleep(1.5)
print(f'launched {len(procs)} cells  pids={[p[0].pid for p in procs.values()]}  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
poll_count = 0
snapshot_every = 3
try:
    while remaining:
        done_this_round = []
        for tag, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                print(f'  [{elapsed:5.1f} min] {tag:>28}: {ok}  json={json_exists}')
                # Auto-dump log on failure so we don't waste a round-trip.
                if rc != 0:
                    print(f'    --- last 50 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-50', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print(f'    --- end log ---')
                done_this_round.append(tag)
        for tag in done_this_round:
            del remaining[tag]
        if remaining:
            poll_count += 1
            if poll_count % snapshot_every == 0:
                snapshot(remaining, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted — SIGKILLing all remaining workers !!!')
    kill_all(remaining)
    raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill — Runtime → Interrupt cell 7 first, THEN run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 8. Copy results + logs to Drive.
import shutil, os
from pathlib import Path
dst_root = '/content/drive/MyDrive/neuro-ai/results/c3_sweep_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
TAGS = ['baseline_lrpull01_n1000','lrpull05_n1000','lrpull10_n1000','lrpull01_n3000','lrpull10_n3000']
for tag in TAGS:
    src = f'reports/c3_sweep_{tag}_2026-05-27'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst_root}/c3_sweep_{tag}_2026-05-27', dirs_exist_ok=True)
if os.path.isdir('reports/c3_sweep_colab_logs'):
    shutil.copytree('reports/c3_sweep_colab_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print('results in', dst_root)
!ls {dst_root}

In [ ]:
# 9. In-notebook headline summary table.
import json
from pathlib import Path

CELLS = [
    ('baseline_lrpull01_n1000', 0.1, 1000),
    ('lrpull05_n1000',          0.5, 1000),
    ('lrpull10_n1000',          1.0, 1000),
    ('lrpull01_n3000',          0.1, 3000),
    ('lrpull10_n3000',          1.0, 3000),
]

STRATA = ('tight', 'spread', 'borderline')

print(f'{"cell":>26} {"lrP":>5} {"n_ev":>5} {"mode":>11} {"stratum":>11}  '
      f'{"std R@K":>22}  {"ctrl R@K":>22}  {"Δ":>7}  disjoint?  n_std  n_ctrl')
for tag, lr_pull, n_events in CELLS:
    p = Path(f'reports/c3_sweep_{tag}_2026-05-27/c3_summary.json')
    if not p.exists():
        print(f'{tag:>26} {lr_pull:>5} {n_events:>5}  MISSING')
        continue
    d = json.loads(p.read_text())
    agg = d['aggregated']
    for mode in d['header']['theta_prime_modes_run']:
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]
            c = agg[mode]['shuffled_control'][stratum]
            dl = agg[mode]['delta_standard_minus_control'][stratum]
            if s['trials'] == 0 and c['trials'] == 0:
                continue
            std_str = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
            ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
            disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
            print(f'{tag:>26} {lr_pull:>5} {n_events:>5} {mode:>11} {stratum:>11}  '
                  f'{std_str:>22}  {ctrl_str:>22}  {dl["delta_recall_at_k"]:>+7.3f}  {disj:>9}  '
                  f'{s["trials"]:>5}  {c["trials"]:>5}')
print()
print('Interpretation guide:')
print('  - If ANY cell shows CI-disjoint=YES on a stratum with both n_std>0 and n_ctrl>0,')
print('    that breaks the Path α null at smoke scale — escalate that cell to n=10.')
print('  - If all cells stay CI-overlapping (typical Δ ~0.0±0.01), Path α remains null;')
print('    next step is Path β (WikiText-2) or Path γ (Phase 3 mechanism redesign).')